# Part 1: Loading, Cleaning and Feature Engineering
**CS316 Project:** Household Energy Consumption Prediction and Carbon Footprint Reduction

**Owner:** Yara

This notebook reads the raw UCI data set, cleans it, aggregates it to hourly energy (kWh) and builds the features used for forecasting. At the end it saves `hourly.csv` and `features.csv`, which the next two notebooks read.

## Setup
Import `pandas` and `numpy`.

In [ ]:
import pandas as pd
import numpy as np

## 1. Importing the Data Set from GitHub
Read the **Individual Household Electric Power Consumption** data set (UCI repository) directly from our GitHub repository. The compressed file (`.txt.xz`) is in the `data` folder and `pd.read_csv` opens it by itself. Replace `YOUR_USERNAME` and `YOUR_REPO` with the real names. If you uploaded the file to Colab yourself, use `urlprefix = ''`.

In [ ]:
urlprefix = 'https://raw.githubusercontent.com/YOUR_USERNAME/YOUR_REPO/main/data/'
dataname = 'household_power_consumption.txt.xz'

The file uses `;` as the separator and `?` for missing values, so `sep` and `na_values` are specified. To save memory, only the needed columns are read (`usecols`) and the numbers are stored as `float32`, which uses half the memory of `float64`.

In [ ]:
num_cols = ['Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity',
            'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
dtypes = {c: 'float32' for c in num_cols}
raw = pd.read_csv(urlprefix + dataname, sep=';', na_values=['?', ''], dtype=dtypes,
                  usecols=['Date', 'Time'] + num_cols, low_memory=False)

In [ ]:
raw.head()

## 2. Inspecting the Data
Check the size of the table and the data type of each column.

In [ ]:
raw.shape

In [ ]:
raw.info()

## 3. Memory Usage
Check how much memory the minute-level table uses.

In [ ]:
print(round(raw.memory_usage(deep=True).sum() / 1e6), 'MB')

## 4. Checking Missing Values
Count the missing values (the `?` entries) in each column and compute the percentage of missing minutes.

In [ ]:
raw.isna().sum()

In [ ]:
print(round(raw['Global_active_power'].isna().mean() * 100, 3), '% of the minutes are missing')

## 5. Creating a Datetime Index
Combine `Date` and `Time` into one datetime column, use it as the index and sort the rows in time order.

In [ ]:
raw['datetime'] = pd.to_datetime(raw['Date'] + ' ' + raw['Time'], format='%d/%m/%Y %H:%M:%S')
raw = raw.drop(columns=['Date', 'Time']).set_index('datetime').sort_index()

In [ ]:
raw.head()

## 6. Filling Short Gaps
Gaps of up to 60 minutes are filled by time interpolation. Longer outages stay missing.

In [ ]:
raw = raw.interpolate(method='time', limit=60)

## 7. Converting Power to Energy
`Global_active_power` is the average power in kW during one minute, so the energy of that minute in kWh is the power divided by 60. The sub-meters are measured in Wh, so they are divided by 1000 to get kWh.

In [ ]:
raw['energy_kwh'] = raw['Global_active_power'] / 60.0
for c in ['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']:
    raw[c] = raw[c] / 1000.0

## 8. Aggregating to Hourly Energy
Sum the minute values to hourly kWh with `resample('h')`. Hours with fewer than 45 valid minutes are unreliable, so they are set to missing and then dropped.

In [ ]:
hourly = raw.resample('h').agg({
    'energy_kwh': 'sum', 'Global_reactive_power': 'mean', 'Voltage': 'mean',
    'Global_intensity': 'mean', 'Sub_metering_1': 'sum', 'Sub_metering_2': 'sum',
    'Sub_metering_3': 'sum'}).astype('float32')

In [ ]:
valid_minutes = raw['Global_active_power'].notna().resample('h').sum()
hourly.loc[valid_minutes < 45, :] = np.nan
print('Hourly rows:', len(hourly), '| missing hours:', hourly['energy_kwh'].isna().sum())

In [ ]:
hourly = hourly.dropna()

## 9. Freeing Memory
The minute-level table is not needed any more, so it is deleted (memory optimization).

In [ ]:
del raw

## 10. Energy Not Covered by the Sub-meters
The three sub-meters cover the kitchen, the laundry room and the water heater with air conditioner. The rest (lighting, refrigerator and other appliances) is the total energy minus the three sub-meters.

In [ ]:
sub_cols = ['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
hourly['Sub_metering_rest'] = (hourly['energy_kwh'] - hourly[sub_cols].sum(axis=1)).clip(lower=0)

In [ ]:
hourly.head()

In [ ]:
hourly.info()

## 11. Summary of Hourly Energy
Use `describe()` to get the count, mean, standard deviation, minimum, quartiles and maximum of the hourly energy.

In [ ]:
hourly['energy_kwh'].describe()

## 12. Building the Features: Calendar Variables
The target is the energy used in the hour. All features use **past information only**. The first group is the calendar: hour, day of week, month and a weekend flag. The sine and cosine of the hour are added so that hour 23 and hour 0 are close to each other.

In [ ]:
df = pd.DataFrame(index=hourly.index)
df['target'] = hourly['energy_kwh']
df['hour'] = df.index.hour
df['dayofweek'] = df.index.dayofweek
df['month'] = df.index.month
df['is_weekend'] = (df.index.dayofweek >= 5).astype('int8')
df['hour_sin'] = np.sin(2 * np.pi * df.index.hour / 24)
df['hour_cos'] = np.cos(2 * np.pi * df.index.hour / 24)

## 13. Building the Features: Lags and Rolling Statistics
Add the energy of earlier hours (lags of 1, 2, 3, 24, 48 and 168 hours; 168 hours is the same hour one week ago) and the rolling mean and standard deviation of the previous hours. The series is shifted by one hour first, so the current hour is never part of its own feature.

In [ ]:
s = hourly['energy_kwh']
for lag in (1, 2, 3, 24, 48, 168):
    df[f'lag_{lag}'] = s.shift(lag)

df['roll_mean_24'] = s.shift(1).rolling(24).mean()
df['roll_std_24'] = s.shift(1).rolling(24).std()
df['roll_mean_168'] = s.shift(1).rolling(168).mean()

## 14. Cleaning the Feature Table
The first rows have no lag values, so they are dropped. The table is stored as `float32` to save memory.

In [ ]:
df = df.astype('float32').dropna()
df.head()

In [ ]:
df.shape

## 15. Saving the Results for the Next Notebooks
Save the hourly table and the feature table. Download both files from the Colab **Files** panel and upload them to the `data` folder of the GitHub repository, so Fai and Aljoharah can read them with `pd.read_csv`.

In [ ]:
hourly.to_csv('hourly.csv')
df.to_csv('features.csv')

In [ ]:
print('hourly.csv:', hourly.shape, '| features.csv:', df.shape)